In [6]:
import os
import requests
import time

# 1. Creamos la estructura de carpetas que exige load_files
os.makedirs('txt-files/libros', exist_ok=True)

print("Iniciando descarga de 100 libros...")
libros_descargados = 0
id_libro = 1

# 2. Bucle que se detiene al llegar a 100
while libros_descargados < 100:
    # URL directa al archivo de texto plano en Gutenberg
    url = f"https://www.gutenberg.org/cache/epub/{id_libro}/pg{id_libro}.txt"
    respuesta = requests.get(url)
    
    # Si el libro existe (código 200), lo guardamos
    if respuesta.status_code == 200:
        ruta = f"txt-files/libros/libro_{id_libro}.txt"
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write(respuesta.text)
            
        libros_descargados += 1
        print(f"Descargado: libro_{id_libro}.txt ({libros_descargados}/100)")
        
        # Pausa de 1 segundo para que el servidor de Gutenberg no te bloquee la IP
        time.sleep(1)
        
    id_libro += 1

print("¡Descarga completada!")

Iniciando descarga de 100 libros...
Descargado: libro_1.txt (1/100)
Descargado: libro_2.txt (2/100)
Descargado: libro_3.txt (3/100)
Descargado: libro_4.txt (4/100)
Descargado: libro_5.txt (5/100)
Descargado: libro_6.txt (6/100)
Descargado: libro_7.txt (7/100)
Descargado: libro_8.txt (8/100)
Descargado: libro_9.txt (9/100)
Descargado: libro_10.txt (10/100)
Descargado: libro_11.txt (11/100)
Descargado: libro_12.txt (12/100)
Descargado: libro_13.txt (13/100)
Descargado: libro_14.txt (14/100)
Descargado: libro_15.txt (15/100)
Descargado: libro_16.txt (16/100)
Descargado: libro_17.txt (17/100)
Descargado: libro_18.txt (18/100)
Descargado: libro_19.txt (19/100)
Descargado: libro_20.txt (20/100)
Descargado: libro_21.txt (21/100)
Descargado: libro_22.txt (22/100)
Descargado: libro_23.txt (23/100)
Descargado: libro_24.txt (24/100)
Descargado: libro_25.txt (25/100)
Descargado: libro_26.txt (26/100)
Descargado: libro_27.txt (27/100)
Descargado: libro_28.txt (28/100)
Descargado: libro_29.txt (29/1

In [12]:
import math
import string
from collections import Counter
from sklearn.datasets import load_files

# Funciones: 
def obtener_matriz_tf(vocabulario, documentos):
    matriz_tf = []
    for doc in documentos:
        conteo = Counter(doc)
        fila_tf = []
        for termino in vocabulario:
            frecuencia = conteo[termino]
            fila_tf.append(frecuencia)
        matriz_tf.append(fila_tf)
    return matriz_tf

def obtener_df(vocabulario, matriz_tf):
    lista_df = []
    num_documentos = len(matriz_tf)
    for j in range(len(vocabulario)):
        contador_docs = 0
        for i in range(num_documentos):
            if matriz_tf[i][j] > 0:
                contador_docs += 1
        lista_df.append(contador_docs)
    return lista_df

def calcular_matriz_tfidf(matriz_tf, lista_df):
    matriz_tfidf = []
    num_documentos = len(matriz_tf)
    num_terminos = len(lista_df)
    for i in range(num_documentos):
        fila_tfidf = []
        for j in range(num_terminos):
            tf = matriz_tf[i][j]
            df_termino = lista_df[j]
            if df_termino > 0:
                idf = math.log10(num_documentos / df_termino)
                tfidf = tf * idf
            else:
                tfidf = 0.0
            fila_tfidf.append(round(tfidf, 4))
        matriz_tfidf.append(fila_tfidf)
    return matriz_tfidf

In [15]:
# Carga de archivos y procesamiento
matriz_tf = ""
lista_df = ""
matriz_tfidf = ""

if __name__ == "__main__":
    # Carga de la carpeta con archivos .txt
    corpus = load_files('txt-files', encoding='utf-8')
    
    # Tomamos todos los documentos crudos del corpus
    textos_crudos = corpus.data[:]
    print(f"Se cargaron {len(textos_crudos)} documentos crudos.")

    # Listas para guardar los datos procesados
    documentos_procesados = []
    vocabulario_set = set() # Usamos un set para evitar palabras duplicadas rápidamente
    
    # Stopwords básicas en inglés (asumiendo que los libros de Gutenberg están en inglés)
    stopwords = {"the", "and", "to", "of", "a", "i", "in", "it", "that", "was", "he", "is", "for", "as", "on", "with"}

    # Limpiamos cada texto
    for i, texto in enumerate(textos_crudos):
        #texto_recortado = texto[:2000] 
        
        # 1. Minúsculas
        texto_limpio = texto.lower()
        
        # 2. Quitar puntuación
        texto_limpio = texto_limpio.translate(str.maketrans('', '', string.punctuation))
        
        # 3. Dividir en lista de palabras (tokenizar)
        palabras = texto_limpio.split()
        
        # 4. Filtrar stopwords y números
        palabras_filtradas = [p for p in palabras if p not in stopwords and p.isalpha()]
        
        documentos_procesados.append(palabras_filtradas)
        vocabulario_set.update(palabras_filtradas)
        
        print(f"Documento procesado: {i+1} de 100.")

    # Convertimos el set a una lista ordenada alfabéticamente
    vocabulario_final = sorted(list(vocabulario_set))
    print(f"El vocabulario final tiene {len(vocabulario_final)} términos únicos.")
    
    # Uso de las funciones
    print("\nCalculando Matriz TF...")
    matriz_tf = obtener_matriz_tf(vocabulario_final, documentos_procesados)
    
    print("Calculando Vector DF...")
    lista_df = obtener_df(vocabulario_final, matriz_tf)
    
    print("Calculando Matriz TF-IDF final...")
    matriz_tfidf = calcular_matriz_tfidf(matriz_tf, lista_df)
    
    print("\n¡Cálculo terminado con éxito!")

Se cargaron 100 documentos crudos.
Documento procesado: 1 de 100.
Documento procesado: 2 de 100.
Documento procesado: 3 de 100.
Documento procesado: 4 de 100.
Documento procesado: 5 de 100.
Documento procesado: 6 de 100.
Documento procesado: 7 de 100.
Documento procesado: 8 de 100.
Documento procesado: 9 de 100.
Documento procesado: 10 de 100.
Documento procesado: 11 de 100.
Documento procesado: 12 de 100.
Documento procesado: 13 de 100.
Documento procesado: 14 de 100.
Documento procesado: 15 de 100.
Documento procesado: 16 de 100.
Documento procesado: 17 de 100.
Documento procesado: 18 de 100.
Documento procesado: 19 de 100.
Documento procesado: 20 de 100.
Documento procesado: 21 de 100.
Documento procesado: 22 de 100.
Documento procesado: 23 de 100.
Documento procesado: 24 de 100.
Documento procesado: 25 de 100.
Documento procesado: 26 de 100.
Documento procesado: 27 de 100.
Documento procesado: 28 de 100.
Documento procesado: 29 de 100.
Documento procesado: 30 de 100.
Documento proc

In [23]:
import pandas as pd

# Convertimos la matriz a un DataFrame con los nombres de las palabras
df_tfidf = pd.DataFrame(matriz_tf, columns=vocabulario_final)

# Mostrar
df_tfidf.iloc[:15, :10]

,aa,aaa,aaanthor,aaargh,aaas,aaasd,aaass,aagcfeaeccspbsu,aage,aaiun
0,0,0,0,0,0,0,0,0,0,0
1,20,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Creamos una tabla con la palabra y la cantidad de documentos donde aparece
df_frecuencias = pd.DataFrame({
    'Término': vocabulario_final,
    'DF (N° de documentos)': lista_df
})

# Mostrar
df_frecuencias.head(15)

,Término,DF (N° de documentos)
0,aa,6
1,aaa,1
2,aaanthor,1
3,aaargh,1
4,aaas,1
5,aaasd,1
6,aaass,1
7,aagcfeaeccspbsu,1
8,aage,1
9,aaiun,4


In [24]:
df_tfidf = pd.DataFrame(matriz_tfidf, columns=vocabulario_final)

# Mostrar
df_tfidf.iloc[:15, :10]

,aa,aaa,aaanthor,aaargh,aaas,aaasd,aaass,aagcfeaeccspbsu,aage,aaiun
0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
1,24.437,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.3979
2,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
3,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
4,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
5,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
6,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
7,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
8,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
9,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0000
